# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AliAlmasri7/FlyRank-ML/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

# 1. My rule and its reason codes

## Rule

My baseline rule identifies content that appears to receive fewer clicks than expected for its observed search position.

For each content item, I compare its observed CTR with the median CTR of other content items in the same position group:

- 1–3
- 4–10
- 11–20
- 21–50
- 50+

The rule calculates a CTR gap:

**CTR gap = expected CTR for the position group − observed CTR**

A larger positive gap means that the content receives less click activity than the typical content in a similar position group.

The baseline score combines:

1. CTR gap — larger gaps receive higher scores.
2. Search impressions — higher impressions increase priority because more searches are affected.

The rule is intended to create a **decision-support queue**, not to predict Google's ranking algorithm or prove that CTR causes ranking changes.

## Reason code

The rule uses one reason code:

**CTR_BELOW_POSITION_PEERS**

This means that the content's measured CTR is below the observed median CTR for its position group.

## Action

The action associated with this reason code is:

**REVIEW_SNIPPET**

The action is a recommendation to review the search-result presentation of the content, such as its title and snippet, because the content is receiving fewer clicks than position peers.

This is an observational baseline. It identifies content for review; it does not establish that changing the snippet will cause traffic or ranking improvements.

In [1]:
import os
import duckdb
import pandas as pd
import numpy as np

# Connect to DuckDB
con = duckdb.connect()

# Hugging Face token
HF_TOKEN = os.getenv("HF_TOKEN")

if HF_TOKEN is None:
    raise ValueError("HF_TOKEN was not found in the environment.")

# Create Hugging Face secret
con.execute(
    f"CREATE OR REPLACE SECRET hf_secret "
    f"(TYPE huggingface, TOKEN '{HF_TOKEN}')"
)

# Dataset location
rel = "hf://datasets/FlyRank/internship-warehouse"

# March 2026 partition
march_path = (
    f"{rel}/fact_content_daily_performance/"
    f"month=2026-03/*.parquet"
)

print("Dataset connected successfully.")
print("Using:", march_path)

Dataset connected successfully.
Using: hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet


In [2]:
march_df = con.sql(f"""
SELECT
    client_hash_id,
    content_hash_id,
    report_date,
    gsc_impressions,
    gsc_clicks,
    gsc_sum_position
FROM read_parquet('{march_path}')
WHERE gsc_data_available IS TRUE
  AND gsc_impressions > 0
""").df()

print("Rows:", len(march_df))
march_df.head()

Rows: 3611061


,client_hash_id,content_hash_id,report_date,gsc_impressions,gsc_clicks,gsc_sum_position
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20,0,67
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1,0,0
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125,1,616
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7,0,28
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11,0,25


In [3]:
content_df = (
    march_df
    .groupby(
        ["client_hash_id", "content_hash_id"],
        as_index=False
    )
    .agg(
        gsc_impressions=("gsc_impressions", "sum"),
        gsc_clicks=("gsc_clicks", "sum"),
        gsc_sum_position=("gsc_sum_position", "sum")
    )
)

# Calculate observed CTR
content_df["observed_ctr"] = (
    content_df["gsc_clicks"] /
    content_df["gsc_impressions"]
)

# Calculate average position
content_df["avg_position"] = np.where(
    content_df["gsc_impressions"] > 0,
    content_df["gsc_sum_position"] /
    content_df["gsc_impressions"],
    np.nan
)

content_df.head()

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,observed_ctr,avg_position
0,client_0797ff3a1fc9a6a5,content_0263d5f9b7a2ecd4,1,0,9,0.000000,9.000000
1,client_0797ff3a1fc9a6a5,content_04c67f3541177192,331,2,4759,0.006042,14.377644
2,client_0797ff3a1fc9a6a5,content_05acc92c165f4386,33,0,309,0.000000,9.363636
3,client_0797ff3a1fc9a6a5,content_0f30e04e709c7b5d,145,0,1178,0.000000,8.124138
4,client_0797ff3a1fc9a6a5,content_1207efddce873942,461,0,6679,0.000000,14.488069


In [4]:
def position_group(position):
    if position <= 3:
        return "1-3"
    elif position <= 10:
        return "4-10"
    elif position <= 20:
        return "11-20"
    elif position <= 50:
        return "21-50"
    else:
        return "50+"

content_df["position_group"] = (
    content_df["avg_position"]
    .apply(position_group)
)

content_df["position_group"].value_counts()

position_group
4-10     83288
21-50    32240
11-20    29922
1-3      18860
50+      12428
Name: count, dtype: int64

In [11]:
position_ctr = (
    content_df
    .groupby("position_group")
    .agg(
        total_impressions=("gsc_impressions", "sum"),
        total_clicks=("gsc_clicks", "sum")
    )
    .reset_index()
)

position_ctr["expected_ctr"] = (
    position_ctr["total_clicks"] /
    position_ctr["total_impressions"]
)

position_ctr

,position_group,total_impressions,total_clicks,expected_ctr
0,1-3,41420140,160562,0.003876
1,11-20,31191659,98488,0.003158
2,21-50,57614383,80635,0.001400
3,4-10,148112436,481189,0.003249
4,50+,2318971,958,0.000413


In [12]:
content_df = content_df.drop(
    columns=["expected_ctr", "ctr_gap"],
    errors="ignore"
)

content_df = content_df.merge(
    position_ctr[["position_group", "expected_ctr"]],
    on="position_group",
    how="left"
)

content_df["ctr_gap"] = (
    content_df["expected_ctr"] -
    content_df["observed_ctr"]
)

# 2. Build the ranked queue

The baseline score prioritizes content where:

- the observed CTR is below the typical CTR for its position group, and
- the content has meaningful search impressions.

This is a prioritization score rather than a prediction score.

The score is designed to answer:

> Which content should be reviewed first based on the observed CTR gap and the amount of search exposure?

The queue contains:

- client_hash_id
- content_hash_id
- impressions
- clicks
- average position
- observed CTR
- position group
- expected CTR
- CTR gap
- score
- reason code
- action
- rank

In [14]:
# Normalize CTR gap between 0 and 1
gap_max = content_df["ctr_gap"].clip(lower=0).max()

if gap_max > 0:
    content_df["gap_score"] = (
        content_df["ctr_gap"].clip(lower=0) / gap_max
    )
else:
    content_df["gap_score"] = 0.0

# Normalize impressions using log1p to reduce the effect of very large values
content_df["impression_score"] = (
    np.log1p(content_df["gsc_impressions"])
)

imp_max = content_df["impression_score"].max()

if imp_max > 0:
    content_df["impression_score"] = (
        content_df["impression_score"] / imp_max
    )
else:
    content_df["impression_score"] = 0.0

# Baseline score
content_df["score"] = (
    0.7 * content_df["gap_score"] +
    0.3 * content_df["impression_score"]
)

# Only positive CTR gaps should receive this action
content_df["reason_code"] = np.where(
    content_df["ctr_gap"] > 0,
    "CTR_BELOW_POSITION_PEERS",
    "NO_ACTION"
)

content_df["action"] = np.where(
    content_df["ctr_gap"] > 0,
    "REVIEW_SNIPPET",
    "NO_ACTION"
)

In [23]:
queue = (
    content_df[
        content_df["reason_code"] == "CTR_BELOW_POSITION_PEERS"
    ]
    .sort_values(
        ["score", "gsc_impressions"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue["rank"] = queue.index + 1

queue.head(10)

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,observed_ctr,avg_position,position_group,gap_score,impression_score,score,reason_code,action,expected_ctr,ctr_gap,rank
0,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984,1,363517,0.000007,2.693038,1-3,0.998089,0.886003,0.964463,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,0.003876,0.003869,1
1,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075,1,38268,0.000008,0.308426,1-3,0.997921,0.879682,0.962449,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,0.003876,0.003868,2
2,client_23a62021009f63c4,content_44f34c0a90047651,212404,24,141435,0.000113,0.665877,1-3,0.970851,0.920004,0.955597,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,0.003876,0.003763,3
3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83834,1,9725,0.000012,0.116003,1-3,0.996923,0.850278,0.952929,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,0.003876,0.003864,4
4,client_23a62021009f63c4,content_bf078007df823490,44707,0,62592,0.000000,1.400049,1-3,1.000000,0.803124,0.940937,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,0.003876,0.003876,5
5,client_1a730cb2640a1abf,content_d61fc394d10cba41,38000,1,89778,0.000026,2.362579,1-3,0.993211,0.790933,0.932528,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,0.003876,0.003850,6
6,client_e547b89c05043229,content_dc91779c3d085398,25625,1,61222,0.000039,2.389151,1-3,0.989933,0.761381,0.921367,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,0.003876,0.003837,7
7,client_fef1a8f436438636,content_66bf45eb0c5bb550,24259,1,50258,0.000041,2.071726,1-3,0.989366,0.757273,0.919738,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,0.003876,0.003835,8
8,client_a80fca3f171ed1de,content_fa17add7836d36c3,12588,0,14366,0.000000,1.141246,1-3,1.000000,0.708070,0.912421,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,0.003876,0.003876,9
9,client_73cda7b4e4f265ea,content_b154f6c2652cfeb9,11344,0,22825,0.000000,2.012077,1-3,1.000000,0.700267,0.910080,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,0.003876,0.003876,10


In [24]:
baseline_queue = queue[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "gsc_sum_position",
        "avg_position",
        "observed_ctr",
        "position_group",
        "expected_ctr",
        "ctr_gap",
        "score",
        "reason_code",
        "action",
        "rank"
    ]
].copy()

baseline_queue.head(10)

,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,gsc_sum_position,avg_position,observed_ctr,position_group,expected_ctr,ctr_gap,score,reason_code,action,rank
0,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984,1,363517,2.693038,0.000007,1-3,0.003876,0.003869,0.964463,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,1
1,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075,1,38268,0.308426,0.000008,1-3,0.003876,0.003868,0.962449,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,2
2,client_23a62021009f63c4,content_44f34c0a90047651,212404,24,141435,0.665877,0.000113,1-3,0.003876,0.003763,0.955597,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,3
3,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83834,1,9725,0.116003,0.000012,1-3,0.003876,0.003864,0.952929,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,4
4,client_23a62021009f63c4,content_bf078007df823490,44707,0,62592,1.400049,0.000000,1-3,0.003876,0.003876,0.940937,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,5
5,client_1a730cb2640a1abf,content_d61fc394d10cba41,38000,1,89778,2.362579,0.000026,1-3,0.003876,0.003850,0.932528,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,6
6,client_e547b89c05043229,content_dc91779c3d085398,25625,1,61222,2.389151,0.000039,1-3,0.003876,0.003837,0.921367,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,7
7,client_fef1a8f436438636,content_66bf45eb0c5bb550,24259,1,50258,2.071726,0.000041,1-3,0.003876,0.003835,0.919738,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,8
8,client_a80fca3f171ed1de,content_fa17add7836d36c3,12588,0,14366,1.141246,0.000000,1-3,0.003876,0.003876,0.912421,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,9
9,client_73cda7b4e4f265ea,content_b154f6c2652cfeb9,11344,0,22825,2.012077,0.000000,1-3,0.003876,0.003876,0.910080,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET,10


In [25]:
from pathlib import Path

output_path = (
    Path("..") /
    "outputs" /
    "baseline_action_score.csv"
)

output_path.parent.mkdir(
    parents=True,
    exist_ok=True
)

baseline_queue.to_csv(
    output_path,
    index=False
)



# 3. Top-10 review

The ranked queue is a prioritization tool, so I manually review the top 20 results.

For each item I record:

- the action,
- why it was selected,
- a confidence note,
- and what could make the recommendation wrong.

The goal is not to prove that every recommendation is correct.

The goal is to identify whether the rule produces plausible decision-support candidates and to understand where the rule can fail.

In [26]:
top10 = baseline_queue.head(10).copy()

top10[
    [
        "rank",
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions",
        "gsc_clicks",
        "avg_position",
        "observed_ctr",
        "expected_ctr",
        "ctr_gap",
        "score",
        "reason_code",
        "action"
    ]
]

,rank,client_hash_id,content_hash_id,gsc_impressions,gsc_clicks,avg_position,observed_ctr,expected_ctr,ctr_gap,score,reason_code,action
0,1,client_73cda7b4e4f265ea,content_8e1334d6356668e3,134984,1,2.693038,0.000007,0.003876,0.003869,0.964463,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET
1,2,client_73cda7b4e4f265ea,content_fec55986a1868d62,124075,1,0.308426,0.000008,0.003876,0.003868,0.962449,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET
2,3,client_23a62021009f63c4,content_44f34c0a90047651,212404,24,0.665877,0.000113,0.003876,0.003763,0.955597,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET
3,4,client_73cda7b4e4f265ea,content_9c057b66c30a3abb,83834,1,0.116003,0.000012,0.003876,0.003864,0.952929,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET
4,5,client_23a62021009f63c4,content_bf078007df823490,44707,0,1.400049,0.000000,0.003876,0.003876,0.940937,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET
5,6,client_1a730cb2640a1abf,content_d61fc394d10cba41,38000,1,2.362579,0.000026,0.003876,0.003850,0.932528,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET
6,7,client_e547b89c05043229,content_dc91779c3d085398,25625,1,2.389151,0.000039,0.003876,0.003837,0.921367,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET
7,8,client_fef1a8f436438636,content_66bf45eb0c5bb550,24259,1,2.071726,0.000041,0.003876,0.003835,0.919738,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET
8,9,client_a80fca3f171ed1de,content_fa17add7836d36c3,12588,0,1.141246,0.000000,0.003876,0.003876,0.912421,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET
9,10,client_73cda7b4e4f265ea,content_b154f6c2652cfeb9,11344,0,2.012077,0.000000,0.003876,0.003876,0.910080,CTR_BELOW_POSITION_PEERS,REVIEW_SNIPPET


## 3. Top-10 review

The following table reviews the top 10 items produced by the baseline rule.

The rule prioritizes content with a large positive CTR gap relative to the aggregated CTR of its position group, with impressions contributing to the priority score.

The recommendations are treated as decision-support candidates rather than confirmed problems. A low observed CTR can have explanations other than the snippet itself, such as query intent, search-result features, or differences in the queries represented by the aggregated data.

| Rank | Action | Why it is here | Confidence note | What would make it wrong |
|---:|---|---|---|---|
| 1 | REVIEW_SNIPPET | 134,984 impressions and observed CTR of 0.000007 versus expected CTR of 0.003876, giving a large CTR gap of 0.003869. | Medium | The low CTR may be explained by query mix or search-result context rather than the snippet. |
| 2 | REVIEW_SNIPPET | 124,075 impressions and observed CTR of 0.000008 versus expected CTR of 0.003876. | Medium | The position-group benchmark may not represent the queries affecting this content. |
| 3 | REVIEW_SNIPPET | 212,404 impressions and observed CTR of 0.000113 versus expected CTR of 0.003876. | Medium | High impressions do not establish that the snippet caused the low CTR. |
| 4 | REVIEW_SNIPPET | 83,834 impressions and observed CTR of 0.000012, substantially below the expected CTR of 0.003876. | Medium | Query intent or SERP features could explain the difference. |
| 5 | REVIEW_SNIPPET | 44,707 impressions with zero observed clicks, producing the maximum CTR gap for its position group. | Low–Medium | Zero clicks may result from the specific queries or search contexts represented in the month. |
| 6 | REVIEW_SNIPPET | 38,000 impressions and observed CTR of 0.000026 versus expected CTR of 0.003876. | Medium | The aggregated position group may contain substantially different query types. |
| 7 | REVIEW_SNIPPET | 25,625 impressions and observed CTR of 0.000039, well below the position-group benchmark. | Medium | The low CTR may have causes unrelated to title or snippet quality. |
| 8 | REVIEW_SNIPPET | 24,259 impressions and observed CTR of 0.000041 versus expected CTR of 0.003876. | Medium | The measured position may not capture all factors affecting click behavior. |
| 9 | REVIEW_SNIPPET | 12,588 impressions and zero observed clicks, resulting in a large CTR gap. | Low–Medium | The result is based on a relatively smaller number of impressions and may be sensitive to the observed month. |
| 10 | REVIEW_SNIPPET | 11,344 impressions and zero observed clicks compared with expected CTR of 0.003876. | Low–Medium | The zero-click observation may not be representative of longer-term performance. |

### Top-10 observation

The strongest candidates are generally pages with both substantial impressions and a large negative difference between observed CTR and the position-group benchmark.

However, several lower-ranked items have zero clicks. Their large CTR gaps can therefore be driven by the benchmark rather than by strong evidence that the content has a snippet problem.

This is an important limitation of the baseline: the score is useful for prioritization, but it does not establish why a page receives fewer clicks or whether a snippet change would improve performance.

The queue should therefore be treated as a **directional decision-support list for human review**, not as an automatic optimization decision.

# 4. Weak picks + leakage check

## Weak picks

Some recommendations may be weak even when they receive a high baseline score.

For example, a high CTR gap does not necessarily mean that the title or snippet is the reason for the low CTR. CTR can vary because of query intent, search-result features, brand recognition, competition, and other factors that are not represented in this baseline.

Therefore, the ranked queue should be treated as a review queue rather than an automatic optimization list.

## Leakage check

This baseline uses only information available in the March 2026 observation window:

- GSC impressions
- GSC clicks
- GSC position
- derived CTR
- position-group benchmark
- derived CTR gap
- impressions-based priority

It does not use:

- future traffic,
- future rankings,
- future labels,
- optimization flags,
- product flags,
- future performance,
- or any label-derived variable.

Therefore, the baseline is an observational decision-support rule rather than a future-outcome prediction.

In [27]:
print("Features used by the baseline:")

baseline_features = [
    "gsc_impressions",
    "gsc_clicks",
    "gsc_sum_position",
    "avg_position",
    "observed_ctr",
    "position_group",
    "expected_ctr",
    "ctr_gap"
]

for feature in baseline_features:
    print("✓", feature)

Features used by the baseline:
✓ gsc_impressions
✓ gsc_clicks
✓ gsc_sum_position
✓ avg_position
✓ observed_ctr
✓ position_group
✓ expected_ctr
✓ ctr_gap


In [28]:
# No future dates beyond March 2026
print(
    "Date range:",
    march_df["report_date"].min(),
    "to",
    march_df["report_date"].max()
)

# Check that the queue contains only positive CTR gaps
print(
    "Negative/zero gaps in queue:",
    (baseline_queue["ctr_gap"] <= 0).sum()
)

# Check ranking
print(
    "Ranks unique:",
    baseline_queue["rank"].is_unique
)

# Check output size
print(
    "Output rows:",
    len(baseline_queue)
)

Date range: 2026-03-01 00:00:00 to 2026-03-31 00:00:00
Negative/zero gaps in queue: 0
Ranks unique: True
Output rows: 140518


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.